# API-Driven Agentic System Showcase

This notebook starts the real backend API, runs orchestration scenarios through `/v1/chat/completions` and `/v1/ingest/documents`, renders traces from `data/runtime`, and fails loudly if any demo scenario misses its expected orchestration behavior.


## 1. Environment And Imports

The notebook resolves the repository root dynamically so it works whether Jupyter starts in the repo root or inside `new_demo_notebook/`.


In [ ]:
from pathlib import Path
import atexit
import sys

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "run.py").exists():
    REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from new_demo_notebook.lib.client import GatewayClient
from new_demo_notebook.lib.preflight import run_preflight
from new_demo_notebook.lib.render import display_coverage_matrix, display_scenario_result
from new_demo_notebook.lib.scenario_runner import ScenarioRunner, load_scenarios, validate_agent_coverage
from new_demo_notebook.lib.server import BackendServerManager

NOTEBOOK_ROOT = REPO_ROOT / "new_demo_notebook"
ARTIFACTS_DIR = NOTEBOOK_ROOT / ".artifacts"
SCENARIOS_PATH = NOTEBOOK_ROOT / "scenarios" / "scenarios.json"
RUNTIME_ROOT = REPO_ROOT / "data" / "runtime"
WORKSPACE_ROOT = REPO_ROOT / "data" / "workspaces"
MEMORY_ROOT = REPO_ROOT / "data" / "memory"

_NOTEBOOK_RESOURCES = {"server": None, "client": None}

def cleanup_resources() -> None:
    client = _NOTEBOOK_RESOURCES.get("client")
    if client is not None:
        try:
            client.close()
        except Exception:
            pass
        _NOTEBOOK_RESOURCES["client"] = None
    server = _NOTEBOOK_RESOURCES.get("server")
    if server is not None:
        try:
            server.stop(force=True)
        except Exception:
            pass
        _NOTEBOOK_RESOURCES["server"] = None

atexit.register(cleanup_resources)

print(f"Repo root: {REPO_ROOT}")
print(f"Scenario manifest: {SCENARIOS_PATH}")
print(f"Runtime root: {RUNTIME_ROOT}")
print(f"Workspace root: {WORKSPACE_ROOT}")
print(f"Memory root: {MEMORY_ROOT}")


## 2. Environment Preflight

Run local preflight checks before the API server starts. This validates runtime path creation, database reachability, provider/model availability, and Docker availability for the data-analyst scenario.


In [ ]:
preflight_report = run_preflight(
    repo_root=REPO_ROOT,
    runtime_root=RUNTIME_ROOT,
    workspace_root=WORKSPACE_ROOT,
    memory_root=MEMORY_ROOT,
)
preflight_rows = preflight_report.to_rows()
preflight_rows

assert preflight_report.ready, "Notebook preflight failed. Inspect preflight_rows before starting the server."


## 3. Start The Backend Server

The notebook launches `python run.py serve-api` in a subprocess and waits for `/health/ready`.


In [ ]:
cleanup_resources()
server = BackendServerManager(repo_root=REPO_ROOT, artifacts_dir=ARTIFACTS_DIR)
base_url = server.start()
client = GatewayClient(base_url)
_NOTEBOOK_RESOURCES["server"] = server
_NOTEBOOK_RESOURCES["client"] = client
model_id = client.get_model_id()

assert model_id, "Gateway returned no model id."
print(f"Server URL: {base_url}")
print(f"Gateway model: {model_id}")
print(f"Server log: {server.log_path}")


## 4. Health, Coverage, And Scenario Inventory

Load the scenario manifest and confirm every runtime agent is covered by at least one demo.


In [ ]:
ready = client.health_ready()
assert ready.get("status") == "ready", ready
print(ready)

scenarios = load_scenarios(SCENARIOS_PATH)
coverage = validate_agent_coverage(scenarios)
display_coverage_matrix(scenarios)
coverage


## 5. Run All Scenarios

Each scenario uses a stable `X-Conversation-ID`, reuses the public API, and renders the resulting session/job traces. The notebook asserts success after every scenario so failures are visible immediately.


In [ ]:
runner = ScenarioRunner(
    client=client,
    repo_root=REPO_ROOT,
    runtime_root=RUNTIME_ROOT,
    workspace_root=WORKSPACE_ROOT,
    memory_root=MEMORY_ROOT,
    model_id=model_id,
)

results = []
for scenario in scenarios:
    print(f"Running {scenario.id} ...")
    result = runner.run_scenario(scenario)
    results.append(result)
    display_scenario_result(result)
    assert result.bundle.session_ids, f"Scenario {scenario.id} produced no persisted session trace."
    result.require_success()

assert all(result.success for result in results), "One or more scenarios failed. Inspect the rendered trace output above."


## 6. Summary Table

This compact summary makes it easy to confirm which route and agents were observed for each scenario.


In [ ]:
summary = [
    {
        "scenario_id": result.scenario.id,
        "success": result.success,
        "observed_route": result.attempts[-1].observed_route,
        "observed_agents": result.attempts[-1].observed_agents,
        "attempts": len(result.attempts),
    }
    for result in results
]
failed = [row for row in summary if not row["success"]]
assert not failed, failed
summary


## 7. Optional: Rerun A Single Scenario

Uncomment the last lines to rerun one scenario interactively.


In [ ]:
scenario_map = {scenario.id: scenario for scenario in scenarios}
scenario_map.keys()
# rerun = runner.run_scenario(scenario_map["coordinator_due_diligence"])
# display_scenario_result(rerun)
# rerun.require_success()


## 8. Cleanup

Stop the backend subprocess when you are done with the demo.


In [ ]:
cleanup_resources()
print("Backend server stopped.")
